# UABR — Phase 1: Environment, model tests, GPU smoke + resume gate
Works on **Colab (T4)** and **Kaggle (GPU T4 x2 / P100)**. Runs in ~15–20 min.

**Gate to pass before Phase 2:** all tests green · 40M model trains in FP16 without NaN · checkpoint resumes with continuous loss · throughput (tokens/s) recorded.

Before running: Colab → *Runtime › Change runtime type › T4 GPU*. Kaggle → *Settings › Accelerator › GPU T4 x2*, *Internet: On*.

## 1. Platform + persistent storage
Colab sessions die without warning, so every checkpoint is mirrored to Google Drive. On Kaggle, `/kaggle/working` is kept when you *Save Version* (later phases also push to the Hugging Face Hub).

In [ ]:
import os, sys
ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle")
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content"
    PERSIST = "/content/drive/MyDrive/uabr"
elif ON_KAGGLE:
    WORK = "/kaggle/working"
    PERSIST = "/kaggle/working/persist"
else:
    WORK, PERSIST = os.getcwd(), os.path.join(os.getcwd(), "persist")
os.makedirs(PERSIST, exist_ok=True)
print("colab" if ON_COLAB else "kaggle" if ON_KAGGLE else "local", "| WORK =", WORK, "| PERSIST =", PERSIST)

## 2. Get the code
Pick **one**: (a) your team GitHub repo, or (b) the `uabr-slm.zip` — on Colab put it in `MyDrive/uabr/`, on Kaggle upload it as a Dataset and add it to the notebook.

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<team>/uabr-slm.git"  (leave empty to use the zip)
import glob, subprocess
os.chdir(WORK)
if REPO_URL:
    if not os.path.exists("uabr-slm"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    zips = glob.glob(f"{PERSIST}/uabr-slm.zip") + glob.glob("/kaggle/input/**/uabr-slm.zip", recursive=True)
    if not zips and ON_COLAB:
        from google.colab import files
        up = files.upload(); zips = [k for k in up if k.endswith(".zip")]
    assert zips, "uabr-slm.zip not found"
    subprocess.run(["unzip", "-oq", zips[0], "-d", WORK], check=True)
os.chdir(f"{WORK}/uabr-slm")
print(os.getcwd()); print(sorted(os.listdir()))

## 3. Install + verify GPU (handbook §6.2)
Torch is preinstalled — do **not** reinstall it.

In [ ]:
!pip install -q pyyaml pytest
!python scripts/check_gpu.py

## 4. Correctness tests (handbook §13)
Parameter count (38–42M), shapes, causal mask, loss masking, tiny-batch overfit, generation/EOS, checkpoint-resume equality. **All 7 must pass.**

In [ ]:
!python -m pytest tests -q

## 5. Synthetic data
The real tokenized corpus comes in Phase 2. Synthetic Markov-chain tokens let us test the *exact* 40M code path now (ideal loss ≈ 1.39).

In [ ]:
!python scripts/make_synthetic_data.py --out_dir data/processed/synthetic --n_train 5000000 --n_val 200000

## 6. Smoke run: real 40M model, FP16, 300 steps (handbook §14)
Watch: loss falling, `grad_norm` finite, `tok_per_s`, `peak_mem_gb`, `data_frac` (fraction of time waiting on data — should be small).

In [ ]:
RUN = "runs/smoke_pretrain"
MIRROR = f"{PERSIST}/smoke_pretrain"
!rm -rf {RUN} {MIRROR}
!python src/train/pretrain.py --config configs/smoke_40m.yaml --max_steps 300 --save_every 100 --log_every 20 --mirror_dir {MIRROR}

## 7. Restart-recovery proof
Resume **from the Drive/persistent copy** (as you would in a fresh session) and run 50 more steps. Loss at step 320+ must continue from ~step 300, not reset to ~9.4.

In [ ]:
!python src/train/pretrain.py --config configs/smoke_40m.yaml --resume {MIRROR} --output_dir {RUN} --max_steps 350 --log_every 10 --mirror_dir {MIRROR}

## 8. Gate report + full-pretraining time estimate

In [ ]:
import json, math
rows = [json.loads(l) for l in open(f"{RUN}/metrics.jsonl")]
tr = [r for r in rows if "loss" in r]; va = [r for r in rows if "val_loss" in r]
steady = [r["tok_per_s"] for r in tr if r["step"] > 40]   # skip warm-up / first-step overhead
tps = sorted(steady)[len(steady)//2]
before = [r for r in tr if r["step"] <= 300][-1]; after = [r for r in tr if r["step"] > 300][0]
hours = 1e9 / tps / 3600
print(f"median throughput   : {tps:,.0f} tokens/s")
print(f"peak GPU memory     : {max(r['peak_mem_gb'] for r in tr)} GB")
print(f"loss step 20 -> 300 : {tr[0]['loss']} -> {before['loss']}")
print(f"resume continuity   : step {before['step']} loss {before['loss']}  ->  step {after['step']} loss {after['loss']}")
print(f"val loss            : {[ (r['step'], r['val_loss']) for r in va]}")
print(f"\n1B-token pretraining estimate: {hours:.1f} GPU-hours  (x1.15 overhead -> {hours*1.15:.1f} h)")
print(f"= about {math.ceil(hours*1.15/11)} Kaggle session(s) of 11 h, or {math.ceil(hours*1.15/3)} Colab chunks of ~3 h")
checks = {
  "loss fell": before["loss"] < tr[0]["loss"] - 1.0,
  "no NaN": all(math.isfinite(r["loss"]) for r in tr),
  "resume continuous": abs(after["loss"] - before["loss"]) < 0.5,
}
for k, v in checks.items(): print(("PASS " if v else "FAIL ") + k)

## 9. Record in the team experiment sheet
Copy: GPU name, torch version, tokens/s, peak memory, estimated hours, PASS/FAIL lines. If anything FAILS, check handbook §14 table (NaN → lower LR / check GradScaler; low throughput → `data_frac`; OOM → reduce `micro_batch_size`, raise `grad_accum_steps`).

**Optional speed test:** add `--compile` to the step-6 command and compare tokens/s (Triton supports T4; usually +20–40%).